# GPT -- Causal Language Modeling

BERT sees both sides. GPT sees only the past. The triangle mask is the most consequential single line of code in modern

## Problem definition

A language model answers one question: give the first t-1 tokens, what is the probability distribution over token k. Train on that signal -- next-token prediction -- and you get a model that can generate arbitrary text one token at a time.

You need each position's prediction to depend only on earlier positions. Otherwise the model trivially cheats by looking at the answer.

The causal mask does this. It is a single upper-triangular matrix of -inf values added to **attention scores** before softmax.

## Basic Concet

Causal mask creates a triangular attention matrix

### The mask

```
[[float(-inf) if j > i else 0.0 for j in range(len)] for i in range(len)]
```

### Parallel training, serial inference

Training:
forward-pass the whole `(N, d_model)`... 
Each mask will produce a single input --> output training item.

Inference:
1. Feed [t1, t2, t3], get[t4]
2. ...
3. Feed [t1, ..., t_{n-1}], get[t_{n}]

That is the autoregressive tax and why decoding is the latency bottleneck for every LLM

### The loss -shift by one

Given tokens [t1, t2, t3, t4]
* Input: [t1, t2, t3]
* Targets: [t2, t3, t4]

for every position i, compute -log P(target_i | inputs[:i+1]) Sum, this is the cross-entropy for the whole sequence.

### Decoding strategies

After training, sampling choices matter more than people think.

| Method | What it does | When to use |
|--------|--------------|-------------|
| Greedy | Argmax every step | Deterministic tasks, code completion |
| Temperature | Divide logits by T, sample | Creative tasks, higher T = more diversity |
| Top-k | Sample from top-k tokens only | Kills low-probability tails |
| Top-p (nucleus) | Sample from smallest set with cumulative prob ≥ p | 2020+ default; adapts to distribution shape |
| Min-p | Keep tokens with `p > min_p * max_p` | 2024+; better at rejecting long tails than top-p |
| Speculative decoding | Draft model proposes N tokens, big model verifies | 2–3× latency reduction at same quality |

In 2026, min-p + temperature 0.7 is a reasonable default for open-weights models. Speculative decoding is table stakes for any production inference stack.


# Build your Own

In [ ]:
import math
import random

def softmax(logits, temperature=1.0):
    if temperature != 1.0:
        logits = [x / temperature for x in logits]
    m = max(logits)
    exps = [math.exp(x - m) for x in logits]
    s = sum(exps)
    return [x / s for x in exps]

def causal_mask(n):
    return [[float("-inf") if j > i else 0.0 for j in range(n)] for i in range(n)]

def attention_scores_with_mask(raw_scores, mask):
    return [
        [s + m for s, m in zip(row, mrow)]
        for row, mrow in zip(raw_scores, mask)
    ]

def apply_softmax_row(row):
    finite = [x for x in row if x != float("-inf")]
    if not finite:
        return [0.0] * len(row)
    
    m = max(finite)
    exps = [math.exp(x - m) if x != float("-inf") else 0.0 for x in row]
    s = sum(exps)
    return [e / s if s > 0 else 0.0 for e in exps]

def cross_entropy_shifted(logits_per_pos, target_ids):
    total = 0.0
    count = 0
    for i in range(len(target_ids) - 1):
        probs = softmax(logits_per_pos[i])
        p = probs[target_ids[i + 1]]
        total += -math.log(max(p, 1e-12))
        count += 1

    return total / count

